In [1]:
!pip install bertopic
!pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.8/88.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.9/56.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5

In [73]:
import pandas as pd
import numpy as np
import torch
import transformers
from bertopic import BERTopic
import os
from sentence_transformers import SentenceTransformer
from umap import UMAP
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
from hdbscan import HDBSCAN
import openai
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI, PartOfSpeech
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [55]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [56]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Read Data

In [57]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
df1 = pd.read_csv('../data/letters_2021_processed.csv')
df1 = df1[['s1_s2', 'full_text', 'LETTER_GENDER']]
df1 = df1.rename(columns={'LETTER_GENDER':'label'})

In [58]:
df2 = pd.read_csv('../data/sentence_sets_trimmed_processed.csv')
df2 = df2[['s1_s2', 'full_text', 'applicant_gender']]
df2 = df2.rename(columns={'applicant_gender':'label'})

In [59]:
df = pd.concat([df1, df2], ignore_index=True)

In [60]:
gender_label_mapping = {
    'F':0,
    'female':0,
    'M':1,
    'male':1
}

In [61]:
df['label'] = df['label'].replace(gender_label_mapping)

# Process Data

In [64]:
letters = df['s1_s2']
sentences = [sent_tokenize(letter) for letter in letters]
sentences = pd.concat([df, pd.Series(sentences, name='sentences')], axis=1)
sentences = sentences[['sentences']].explode('sentences')
df_sentences = pd.merge(df, sentences, left_index=True, right_index=True)

# Topic Modeling

In [68]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(sentences['sentences'].tolist(), show_progress_bar=True)

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

hdbscan_model = HDBSCAN(min_cluster_size=150, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4848 [00:00<?, ?it/s]

In [74]:
# KeyBERT
keybert_model = KeyBERTInspired()

# Part-of-Speech
pos_model = PartOfSpeech("en_core_web_sm")

# MMR
mmr_model = MaximalMarginalRelevance(diversity=0.3)

# GPT-3.5
client = openai.OpenAI(api_key="sk-...")
prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]
The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short but highly descriptive topic label of at most 5 words. Make sure it is in the following format:
topic: <topic label>
"""
openai_model = OpenAI(client, model="gpt-3.5-turbo", exponential_backoff=True, chat=True, prompt=prompt)

# All representation models
representation_model = {
    "KeyBERT": keybert_model,
    # "OpenAI": openai_model,  # Uncomment if you will use OpenAI
    "MMR": mmr_model,
    "POS": pos_model
}

In [76]:
topic_model = BERTopic(

  # Pipeline models
  embedding_model=embedding_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  representation_model=representation_model,

  # Hyperparameters
  top_n_words=10,
  verbose=True
)

# Train model
topics, probs = topic_model.fit_transform(df_sentences['sentences'].tolist(), embeddings)



2025-02-04 16:52:01,543 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-04 16:57:33,924 - BERTopic - Dimensionality - Completed ✓
2025-02-04 16:57:33,930 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-04 16:58:10,427 - BERTopic - Cluster - Completed ✓
2025-02-04 16:58:10,460 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-04 16:58:38,160 - BERTopic - Representation - Completed ✓


# Analyze Topics

In [122]:
topic_info = topic_model.get_topic_info()

In [126]:
topic_info

,Topic,Count,Name,Representation,KeyBERT,MMR,POS,Representative_Docs
0,-1,62742,-1_identifier_medical_clinical_work,"[identifier, medical, clinical, work, patients...","[dr identifier, medical student, identifier, m...","[medical, clinical, work, patients, research, ...","[identifier, medical, clinical, work, patients...",[i have personally known dr. identifier sinc...
1,0,5930,0_patients_patient_care_staff,"[patients, patient, care, staff, rapport, diff...","[patients families, patients staff, patients, ...","[patients, staff, compassion, patient care, pa...","[patients, patient, care, staff, rapport, diff...",[they developed great rapport with patients a...
2,1,5719,1_anesthesiology_anesthesiology residency_anes...,"[anesthesiology, anesthesiology residency, ane...","[identifier anesthesiology, anesthesiology ide...","[anesthesiology, anesthesiology residency, ane...","[anesthesiology, anesthesiologist, residency, ...",[it is my pleasure to recommend identifier ...
3,2,5530,2_anesthesia_rotation_anesthesia residency_ane...,"[anesthesia, rotation, anesthesia residency, a...","[anesthesia identifier, identifier anesthesia,...","[anesthesia, anesthesia residency, anesthesiol...","[anesthesia, rotation, anesthesiology, identif...",[identifier during their anesthesia electiv...
4,3,4182,3_anesthesiology_anesthesiologist_career_field,"[anesthesiology, anesthesiologist, career, fie...","[excellent anesthesiologist, future anesthesio...","[anesthesiology, anesthesiologist, career anes...","[anesthesiology, anesthesiologist, career, fie...",[i think they will make an excellent anesthe...
...,...,...,...,...,...,...,...,...
98,97,177,97_perioperative_surgical_perioperative care_p...,"[perioperative, surgical, perioperative care, ...","[perioperative patients, patients perioperativ...","[perioperative care, perioperative period, per...","[perioperative, surgical, perioperative care, ...",[they was able to identify relevant comorbidi...
99,98,172,98_enjoys_music_hobbies_spare time,"[enjoys, music, hobbies, spare time, playing, ...","[enjoys sports, enjoys playing, time enjoys, h...","[enjoys, music, hobbies, spare time, running, ...","[music, hobbies, spare time, spare, hiking, mu...",[in their spare time identifier enjoys spo...
100,99,162,99_late_stayed_early_stayed late,"[late, stayed, early, stayed late, early staye...","[arrived early, stayed late, early stayed, arr...","[stayed late, early stayed, arrived early, sta...","[late, early, day, time, ready, morning, willi...","[they arrived early and stayed late ., they ..."
101,100,157,100_navy_flight_officer_air,"[navy, flight, officer, air, military, marine,...","[flight surgeons, awarded navy, flight surgeon...","[navy, officer, military, flight surgeon, air ...","[navy, flight, officer, air, military, marine,...",[right now they is finishing their tour as...


In [113]:
topic_model.get_topic(70, full=True)

{'Main': [('clerkship', 0.08696053211585056),
  ('pass', 0.08647035219248585),
  ('clerkships', 0.06359942191794754),
  ('honors', 0.05526009984595896),
  ('grades', 0.04270938518398813),
  ('high pass', 0.04002740961988317),
  ('received', 0.03513875259950373),
  ('grades pass', 0.031161937050353233),
  ('pass fail', 0.030230728483702254),
  ('clinical clerkships', 0.02962995607848971)],
 'KeyBERT': [('grades pass', 0.5808706),
  ('clinical clerkships', 0.5458611),
  ('clinical honors', 0.5379102),
  ('medicine clerkship', 0.4873869),
  ('grades', 0.47752982),
  ('science courses', 0.47528782),
  ('grades honors', 0.47456256),
  ('honors clerkship', 0.47127682),
  ('surgical clerkship', 0.44107553),
  ('courses', 0.438328)],
 'MMR': [('clerkship', 0.08696053211585056),
  ('clerkships', 0.06359942191794754),
  ('honors', 0.05526009984595896),
  ('grades pass', 0.031161937050353233),
  ('clinical clerkships', 0.02962995607848971),
  ('science courses', 0.023990531122282638),
  ('medicin

In [104]:
topic_df = pd.concat([df_sentences['sentences'].reset_index(drop=True), pd.Series(topics, name='topic')], axis=1, ignore_index=True)

In [105]:
result = pd.concat([topic_df, df_sentences.reset_index(drop=True)], axis=1)

In [117]:
# result[result[1] == 70]['label'].value_counts()

In [118]:
# topics_per_class = topic_model.topics_per_class(df_sentences['sentences'].tolist(), classes=df_sentences['label'].tolist())

In [119]:
# topic_model.visualize_topics_per_class(topics_per_class)
